# 05 — Review FP32 vs INT8 runtime results (updated)

This notebook **does not run models**. It reads previously completed benchmark files and creates comparison CSVs and figures.

The new summary script makes distinct matched comparisons:

1. `fp32_eager` vs `int8_eager` — **original INT8** quantization.
2. `fp32_compiled` vs `int8_compiled` — original INT8, only if that benchmark completed (it previously failed).
3. `fp32_compiled` vs **`int8_compiled_fp32_head`** — **modified INT8** with the final classifier in FP32.

A compiled comparison is **not** evidence for the original INT8 behavior unless prediction-equivalence checks support it.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import pandas as pd

def find_analysis_folder():
    here = Path.cwd().resolve()
    for directory in [here, *here.parents]:
        for candidate in (directory / 'notebooks/runtime_analysis', directory / 'runtime_analysis', directory):
            candidate = candidate.resolve()
            if (candidate / 'config.json').is_file() and (candidate / 'worker.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate notebooks/runtime_analysis. Check notebook location.')

ANALYSIS_DIR = find_analysis_folder()
REPO_ROOT = ANALYSIS_DIR.parent.parent
CONFIG = json.loads((ANALYSIS_DIR / 'config.json').read_text(encoding='utf-8'))
RESULTS_ROOT = REPO_ROOT / CONFIG['results_dir']
print('Analysis directory:', ANALYSIS_DIR)
print('Project root:', REPO_ROOT)
print('Results directory:', RESULTS_ROOT)

import numpy as np
import matplotlib.pyplot as plt

## 1 — Consolidate successful benchmark files

Stale `benchmark_summary.json` files from variants with a later recorded failure are excluded. The new helper saves:

- `all_benchmark_variants.csv`
- `all_throughput_repeats.csv`
- `all_latency_samples.csv`
- `paired_fp32_int8_comparison.csv` (only eligible matched runs)
- `benchmark_inclusion_status.csv`
- `pair_eligibility_status.csv`

A pair is excluded if it differs in key benchmarking conditions, including the final incomplete-batch policy or compilation mode.

In [ ]:
subprocess.run(
    [sys.executable, str(ANALYSIS_DIR / 'summarize_results.py')],
    cwd=REPO_ROOT, check=True,
)

include = pd.read_csv(RESULTS_ROOT / 'benchmark_inclusion_status.csv')
print('Benchmark inclusion status:')
display(include)

all_path = RESULTS_ROOT / 'all_benchmark_variants.csv'
if not all_path.exists():
    raise FileNotFoundError('No successful benchmark summaries were found.')

all_variants = pd.read_csv(all_path)
base_cols = [
    'seed', 'variant', 'device', 'compile_mode', 'num_hans_examples',
    'partial_batch_policy', 'serialized_state_mib',
    'cuda_model_resident_delta_mib', 'median_peak_cuda_allocated_mib',
    'median_throughput_examples_per_second', 'median_full_hans_seconds',
    'batch1_median_forward_latency_ms', 'batch32_median_forward_latency_ms',
]
display(all_variants[[c for c in base_cols if c in all_variants.columns]])

## 2 — Evaluate which comparisons are eligible

We only attribute performance differences to quantization when FP32 and INT8 runs share the same execution mode and input/benchmark conditions. **The FP32-head INT8 configuration is named explicitly** rather than being confused with the original W8A8 model.

In [ ]:
eligibility = pd.read_csv(RESULTS_ROOT / 'pair_eligibility_status.csv')
display(eligibility)
paired_path = RESULTS_ROOT / 'paired_fp32_int8_comparison.csv'
if paired_path.exists():
    paired = pd.read_csv(paired_path)
    show = [
        'seed', 'mode', 'fp32_variant', 'int8_variant',
        'fp32_median_throughput_examples_per_second',
        'int8_median_throughput_examples_per_second',
        'int8_over_fp32_throughput_x',
        'fp32_median_peak_cuda_allocated_mib',
        'int8_median_peak_cuda_allocated_mib',
    ]
    display(paired[[col for col in show if col in paired.columns]])
else:
    paired = pd.DataFrame()
    print('No eligible FP32 vs INT8 pairs yet. Check pair_eligibility_status.csv.')

## 3 — Visualize available variants

These graphs show **observed data** for every successful variant. Not all bars constitute fair pairwise comparisons; use the paired table above for quantitative claims.

Any metric absent from the completed configurations is skipped with a message, not treated as zero.

In [ ]:
def plot_available_metric(frame, field, ylabel, filename, title):
    if field not in frame.columns:
        print(f'SKIP {field}: not recorded in the existing results.')
        return
    rows = frame[['seed', 'variant', field]].copy()
    rows[field] = pd.to_numeric(rows[field], errors='coerce')
    rows = rows.dropna(subset=[field])
    if rows.empty:
        print(f'SKIP {field}: no valid measurements available.')
        return
    pivot = rows.pivot(index='seed', columns='variant', values=field)
    ax = pivot.plot.bar(figsize=(10, 5), rot=0)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Seed')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / filename, dpi=180, bbox_inches='tight')
    plt.show()

plot_available_metric(all_variants, 'serialized_state_mib', 'MiB',
                      'runtime_serialized_state.png', 'Serialized model-state size')
plot_available_metric(all_variants, 'cuda_model_resident_delta_mib', 'MiB',
                      'runtime_resident_cuda_allocation.png', 'Loaded-model CUDA allocation delta')
plot_available_metric(all_variants, 'median_peak_cuda_allocated_mib', 'MiB',
                      'runtime_peak_cuda_allocated.png', 'Peak allocated CUDA tensor memory')
plot_available_metric(all_variants, 'median_throughput_examples_per_second', 'Examples/s',
                      'runtime_throughput_comparison.png', 'HANS throughput (batch 32)')
plot_available_metric(all_variants, 'batch32_median_forward_latency_ms', 'ms/batch',
                      'runtime_batch32_latency.png', 'Batch-32 forward latency')
plot_available_metric(all_variants, 'batch1_median_forward_latency_ms', 'ms/example',
                      'runtime_batch1_latency.png', 'Batch-1 forward latency (only where measured)')

## 4 — Prediction equivalence and research interpretation

Read the output from notebook 04. Specifically inspect `int8_compiled_fp32_head` vs both `int8_eager` and the original saved INT8 predictions.

If a prediction file is missing or comparison failed, **do not claim equivalence**. If the compiled INT8 classifier-head variant changes HANS predictions, report its behavioral performance separately from the original full-W8A8 model.

In [ ]:
checks_file = RESULTS_ROOT / 'prediction_equivalence_checks.csv'
status_file = RESULTS_ROOT / 'prediction_equivalence_status.csv'
if status_file.exists():
    print('Prediction comparisons attempted:')
    display(pd.read_csv(status_file))
else:
    print('No prediction-equivalence status. Run notebook 04 first.')

if checks_file.exists():
    checks = pd.read_csv(checks_file)
    display(checks)
    focus = checks[checks['candidate_variant'].eq('int8_compiled_fp32_head')]
    print('Checks involving the new INT8 FP32-head model:')
    if focus.empty:
        print('No completed equivalence tests — cannot transfer original HANS behavioral metrics.')
    else:
        display(focus[['seed', 'reference_variant', 'candidate_variant',
                       'mnli_3way_disagreement_n', 'hans_binary_disagreement_n',
                       'probability_mae', 'all_probabilities_within_tolerance']])
else:
    print('Prediction-equivalence results not generated yet.')

print('All generated CSVs and figures are under:', RESULTS_ROOT.resolve())

## Reporting guidance

**Core metrics:** serialized model-state size, loaded and peak allocated CUDA tensor memory, batch-32 throughput, and batch-1 latency *only if actually measured with the same policy*. Total HANS inference time is useful supporting information but is mathematically related to throughput.

**Experimental qualifiers:** compiled INT8 with an FP32 classification head is a separate deployment configuration; optional final-batch duplication changes physical compute by 16 examples on the 30,000-row HANS workload; comparisons of eager and compiled execution are not purely quantization effects. Record compilation mode and padding policy with every figure/table.